In [ ]:
df_agg = dfFinal.groupby(["Data da Operação", "Tipo Operação", "Ativo"]).agg(
    Qtde=("Qtde", "sum"),
    Preco=("Preço (R$)", "mean"),
    Custo_medio=("Custo Médio (R$)", "sum"),
    Perc=("%", "mean"),
    Valor_Compra=("Valor Compra", "sum")
).sort_values(by=["Data da Operação", "Tipo Operação"], ascending=[True, False])

In [ ]:
df_reset = df_agg.reset_index()
df_reset["Tipo Operação"] = df_reset["Tipo Operação"].str.replace("Venda (planilha)","Venda", regex=False)

In [ ]:
compras = df_reset[df_reset["Tipo Operação"] == 'Compra'].copy().reset_index(drop=True)
vendas  = df_reset[df_reset["Tipo Operação"] == 'Venda'].copy().reset_index(drop=True)

In [ ]:
vendas.rename(columns={"Valor_Compra": "Valor_Venda",
                       "Tipo Operação": "Operação Venda",
                       "Ativo": "Ativo Venda"
                      }, inplace=True)
vendas.drop(columns=["Data da Operação"], inplace=True)

In [ ]:
df_result = pd.concat([compras, vendas], axis=1)
df_result["Result"] = df_result["Valor_Venda"] - df_result["Valor_Compra"]

In [ ]:
df_result["Data da Operação"] = pd.to_datetime(df_result["Data da Operação"])

df_tot_mes_daytrade = (
    df_result
    .groupby(df_result["Data da Operação"].dt.to_period("M"))["Result"]
    .sum()
    .reset_index()
)
df_tot_mes_daytrade.index = pd.RangeIndex(start=1, stop=len(df_tot_mes_daytrade) + 1, step=1)
resultado = round(df_tot_mes_daytrade['Result'].sum(), 2)
df_tot_mes_daytrade.loc["Total"] = ['== Ano ==', resultado]

In [ ]:
diff = df_result['Valor_Venda'] - df_result['Valor_Compra']
df_result['lucro'] = (diff > 0).astype(int)
df_result['preju'] = (diff < 0).astype(int)